# Лабораторная работа №11: Alignment VLM по предпочтениям

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Собрать preference pairs без length/style shortcut.
- Провести LoRA preference optimization на 2B fallback.
- Сравнить base/SFT/aligned по win rate и hallucinations.


## Теоретическая часть

Preference optimization использует пары chosen/rejected. DPO увеличивает относительную вероятность chosen относительно reference policy без отдельной reward model. Для VLM важно, чтобы изображения и prompts совпадали в обеих ветвях.


**Пара предпочтений.** Тройка данных: вход (изображение и вопрос), выбранный и отвергнутый ответы. В обеих ветвях вход одинаковый, чтобы модель училась сравнивать ответы, а не изображения. Предпочтения берутся из разметки датасета, а не из экспертных оценок полезности.

**DPO (прямая оптимизация предпочтений).** Метод обучения на парах предпочтений без отдельной модели вознаграждения. Обучаемая модель сравнивается с замороженной опорной моделью. Функция потерь поощряет преимущество выбранного ответа над отвергнутым.

**Log-prob (логарифм вероятности).** Показатель вероятности ответа при заданном входе. Вопрос и ответ токенизируются вместе, причём вопрос должен совпадать с началом полной последовательности. Позиция \(t\) предсказывает токен \(t+1\). Учитываются только токены ответа. Для обучаемой модели используются градиенты, для опорной — нет.

**LoRA (низкоранговое дообучение).** Метод, при котором вместо всех весов обучаются только небольшие дополнительные матрицы в линейных слоях. Это доли процента параметров. Базовая модель загружается в 4-битном формате для экономии памяти.

**Три модели при оценке.**
- Foundation — базовая модель без адаптера.
- Reference — модель с адаптером из работы 9 после дообучения на инструкциях.
- Aligned — модель после выравнивания.

**EM (точное совпадение).** Доля ответов, в которых буква совпадает с эталонной. Измеряет точность, а не предпочтения. Дополнительно оцениваются формат, предпочтения и галлюцинации. Незаполненные суждения исключаются из подсчёта.

### Математическая постановка

$$\mathcal L_{DPO}=-\log\sigma\left(\beta[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}]\right).$$


### Материалы

- [DPO paper](https://arxiv.org/abs/2305.18290)
- [TRL цикл оптимизации DPO](https://huggingface.co/docs/trl/dpo_trainer)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Preference dataset является экспертным входом; пример с несуществующим cats.jpg не используется. Нужен адаптер lab 9 и GPU. При смене версии TRL сверьте мультимодальный цикл оптимизации DPO и зафиксируйте версии. `FAST_DEV_RUN=True` выполняет реальные два шага DPO, но не является evidence качества. Без заполненного слепого экспертного аудита итоговая метрика не создаётся.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 99
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


In [2]:
# Библиотеки и вспомогательные функции

import torch
import shutil
import zipfile
from PIL import Image
from collections import Counter

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig
)
from peft import PeftModel, PeftConfig, LoraConfig, get_peft_model
from datasets import Dataset

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_DIR = ARTIFACTS / "lab9_adapter"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (
    torch.cuda.is_available() and torch.cuda.is_bf16_supported()
) else torch.float16 if torch.cuda.is_available() else torch.float32

random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
import torch.nn.functional as F
from torch.optim import AdamW

import math
import time
import re
import shutil
import subprocess
from collections import Counter

In [4]:
!apt-get -qq install unrar
!mkdir -p artifacts/lab9_adapter
!unrar x -o+ artifacts/lab9_adapter.rar artifacts/lab9_adapter/


UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from artifacts/lab9_adapter.rar

Extracting  artifacts/lab9_adapter/trainer_log.jsonl                       0%  OK 
Extracting  artifacts/lab9_adapter/trainer_state.json                      0%  OK 
Extracting  artifacts/lab9_adapter/training_args.bin                       0%  OK 
Extracting  artifacts/lab9_adapter/adapter_config.json                     0%  OK 
Extracting  artifacts/lab9_adapter/adapter_model.safetensors               7% 14% 22% 29% 36% 44% 51% 58% 66% 73% 80% 88% 95% 96%  OK 
Extracting  artifacts/lab9_adapter/all_results.json                       96%  OK 
Extracting  artifacts/lab9_adapter/chat_template.jinja                    96%  OK 
Extracting  artifacts/lab9_adapter/eval_results.json                      96%  OK 
Extracting  artifacts/lab9_adapter/processor_co

## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Предпочтения из визуальных ответов

*Вес: 1 балл.*

**Постановка.** Подготовьте ScienceQA через открытый загрузчик. Для каждого изображения chosen — правильная буква, rejected — следующий неверный вариант. Явно пометьте: это предпочтения на основе эталонной разметки, а не экспертные оценки полезности.

**Результат.** Список пар с provenance.

**Критерий принятия.** Изображения существуют, ответы различаются и соответствуют вариантам; лицензия и источник сохранены.


In [5]:
!pip install rarfile

In [4]:
# --------------------------------------------------
# 1. Пути к датасету
# --------------------------------------------------

DATA_DIR = Path("data")
ARCHIVE_PATH = DATA_DIR / "lab9.rar"
EXTRACT_DIR = DATA_DIR / "lab9"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Распаковка архива
# --------------------------------------------------

manifest_path = EXTRACT_DIR / "manifest.jsonl"

if not manifest_path.exists():
    import rarfile

    rar = rarfile.RarFile(ARCHIVE_PATH)
    try:
        rar.extractall(DATA_DIR)
    except rarfile.RarCannotExec:
        raise RuntimeError(
            "Для распаковки RAR5 установите unrar или 7-Zip "
            "и повторно запустите ячейку."
        )

assert manifest_path.exists(), (
    f"Не найден manifest.jsonl: {manifest_path}"
)

# --------------------------------------------------
# 3. Загрузка manifest
# --------------------------------------------------

records = []

with manifest_path.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print("Всего записей в manifest:", len(records))

if not records:
    raise ValueError("Manifest не содержит записей.")

print("Поля первой записи:", list(records[0].keys()))

# --------------------------------------------------
# 4. Индекс изображений
# --------------------------------------------------

image_files = list(EXTRACT_DIR.rglob("*.png"))

# Также учитываем JPG/JPEG
image_files += list(EXTRACT_DIR.rglob("*.jpg"))
image_files += list(EXTRACT_DIR.rglob("*.jpeg"))

images_by_name = {
    p.name: p for p in image_files
}

images_by_stem = {
    p.stem: p for p in image_files
}

print("Найдено изображений:", len(image_files))

# --------------------------------------------------
# 5. Формирование preference pairs
# --------------------------------------------------

preferences = []

missing_images = 0
invalid_records = 0

for row in records:

    # Используем только обучающую выборку.
    if row.get("split") != "train":
        continue

    choices = row.get("choices")
    correct_idx = row.get("answer_index")

    if not choices or correct_idx is None:
        invalid_records += 1
        continue

    correct_idx = int(correct_idx)

    if not (0 <= correct_idx < len(choices)):
        invalid_records += 1
        continue

    # Правильный ответ — chosen.
    chosen_idx = correct_idx

    # Следующий неверный вариант — rejected.
    rejected_idx = (correct_idx + 1) % len(choices)

    if rejected_idx == chosen_idx:
        invalid_records += 1
        continue

    chosen = chr(65 + chosen_idx)
    rejected = chr(65 + rejected_idx)

    if chosen == rejected:
        invalid_records += 1
        continue

    # Поиск соответствующего изображения.
    image_value = row.get("image")

    if not image_value:
        image_value = row.get("image_path")

    if not image_value:
        missing_images += 1
        continue

    image_value = str(image_value)

    image_path = images_by_name.get(
        Path(image_value).name
    )

    if image_path is None:
        image_path = images_by_stem.get(
            Path(image_value).stem
        )

    if image_path is None:
        missing_images += 1
        continue

    # Вычисление SHA256 изображения.
    image_sha256 = hashlib.sha256(
        image_path.read_bytes()
    ).hexdigest()

    preferences.append({
        "id": row.get("id", image_path.stem),
        "image": str(image_path.resolve()),
        "question": row.get("question", ""),
        "choices": choices,

        "chosen": chosen,
        "rejected": rejected,

        "answer_index": chosen_idx,
        "rejected_index": rejected_idx,

        "image_sha256": image_sha256,

        "preference_source": "dataset_ground_truth",
        "preference_type": (
            "label_based_not_expert_utility"
        ),

        "dataset": "lab9",
        "source_url": row.get("source_url"),
        "license": row.get("license", "See ATTRIBUTION.md")
    })

# --------------------------------------------------
# 6. Проверки корректности
# --------------------------------------------------

assert preferences, (
    "Не сформировано ни одной пары. "
    "Проверь структуру manifest и пути изображений."
)

assert all(
    p["chosen"] != p["rejected"]
    for p in preferences
)

assert all(
    Path(p["image"]).is_file()
    for p in preferences
)

assert all(
    p["chosen"] in [
        chr(65 + i) for i in range(len(p["choices"]))
    ]
    and p["rejected"] in [
        chr(65 + i) for i in range(len(p["choices"]))
    ]
    for p in preferences
)

# --------------------------------------------------
# 7. Сохранение результата
# --------------------------------------------------

preferences_path = ARTIFACTS / "preferences_raw.jsonl"

with preferences_path.open(
    "w", encoding="utf-8"
) as f:
    for p in preferences:
        f.write(
            json.dumps(p, ensure_ascii=False) + "\n"
        )

# --------------------------------------------------
# 8. Итоговая статистика
# --------------------------------------------------

print("\nPreference pairs:", len(preferences))
print("Изображения найдены:", len(preferences))
print("Отсутствующие изображения:", missing_images)
print("Некорректные записи:", invalid_records)

print(
    "Chosen и rejected различаются:",
    all(p["chosen"] != p["rejected"] for p in preferences)
)

print("Источник предпочтений: dataset ground truth")
print("Тип: label-based, not expert utility")
print("Сохранено:", preferences_path)


Всего записей в manifest: 768
Поля первой записи: ['id', 'split', 'image', 'question', 'answer', 'choices', 'answer_index', 'label_type', 'license', 'source_url', 'subject', 'grade', 'image_sha256']
Найдено изображений: 768

Preference pairs: 512
Изображения найдены: 512
Отсутствующие изображения: 0
Некорректные записи: 0
Chosen и rejected различаются: True
Источник предпочтений: dataset ground truth
Тип: label-based, not expert utility
Сохранено: artifacts/preferences_raw.jsonl


### Отложенный тест и смещения пар

*Вес: 1 балл.*

**Постановка.** Используйте official train для DPO и ID test для итоговой оценки. Сверьте ID и хеши изображений. Посчитайте распределение типов ошибок и долю случаев, где chosen длиннее rejected.

**Результат.** preferences.jsonl и pair_audit.json.

**Критерий принятия.** Нет пересечений; длина ответа не используется как скрытый эталон правильности.


In [5]:

processor = AutoProcessor.from_pretrained(MODEL_ID)

# --------------------------------------------------
# 1. Разделение датасета по выборкам
# --------------------------------------------------

SPLITS = (
    "train",
    "validation",
    "id_test",
    "ood_test",
    "general_test"
)

split_records = {
    split: [r for r in records if r["split"] == split]
    for split in SPLITS
}

train_records = split_records["train"]

test_splits = {
    split: split_records[split]
    for split in SPLITS
    if split != "train"
}

# --------------------------------------------------
# 2. Проверка пересечений ID и изображений
# --------------------------------------------------

train_ids = {r["id"] for r in train_records}

train_hashes = {
    r["image_sha256"]
    for r in train_records
    if r.get("image_sha256")
}

overlap_audit = {}

for split, test_records in test_splits.items():

    test_ids = {r["id"] for r in test_records}

    test_hashes = {
        r["image_sha256"]
        for r in test_records
        if r.get("image_sha256")
    }

    id_overlap = train_ids & test_ids
    image_overlap = train_hashes & test_hashes

    overlap_audit[split] = {
        "id_overlap_count": len(id_overlap),
        "image_hash_overlap_count": len(image_overlap),
        "id_overlap": list(id_overlap),
        "image_hash_overlap": list(image_overlap)
    }

    assert not id_overlap, (
        f"Обнаружено пересечение ID между train и {split}: "
        f"{len(id_overlap)}"
    )

    assert not image_overlap, (
        f"Обнаружено пересечение изображений между train и {split}: "
        f"{len(image_overlap)}"
    )

print("Проверка пересечений пройдена для всех тестовых выборок.")

# --------------------------------------------------
# 3. Формирование preference pairs только для train
# --------------------------------------------------

train_preferences = [
    p for p in preferences
    if p["id"] in train_ids
]

assert train_preferences, "Нет обучающих preference pairs."

# Проверяем отсутствие тестовых ID в DPO
for split, test_records in test_splits.items():

    test_ids = {r["id"] for r in test_records}

    leaked_ids = {
        p["id"] for p in train_preferences
    } & test_ids

    assert not leaked_ids, (
        f"Утечка данных из {split} в DPO!"
    )

# --------------------------------------------------
# 4. Анализ длины ответов
# --------------------------------------------------

lengths = [
    {
        "chosen": len(
            processor.tokenizer.encode(
                p["chosen"],
                add_special_tokens=False
            )
        ),
        "rejected": len(
            processor.tokenizer.encode(
                p["rejected"],
                add_special_tokens=False
            )
        )
    }
    for p in train_preferences
]

chosen_longer = sum(
    x["chosen"] > x["rejected"] for x in lengths
)

rejected_longer = sum(
    x["rejected"] > x["chosen"] for x in lengths
)

equal_length = (
    len(lengths) - chosen_longer - rejected_longer
)

error_distribution = Counter(
    p["rejected"] for p in train_preferences
)

# --------------------------------------------------
# 5. Формирование аудита
# --------------------------------------------------

pair_audit = {
    "dataset": "lab9",
    "preference_source": (
        "ground_truth_labels_not_expert_judgments"
    ),

    "train_count": len(train_records),

    "test_counts": {
        split: len(test_records)
        for split, test_records in test_splits.items()
    },

    "preference_pairs_count": len(train_preferences),

    "overlap_checks": overlap_audit,

    "rejected_option_distribution": dict(error_distribution),

    "chosen_longer_count": chosen_longer,
    "rejected_longer_count": rejected_longer,
    "equal_length_count": equal_length,

    "chosen_longer_fraction": (
        chosen_longer / len(lengths) if lengths else 0
    ),

    "length_shortcut": False,

    "length_interpretation": (
        "Ответы представлены буквами одинакового формата; "
        "длина не используется для назначения предпочтения."
    )
}

# --------------------------------------------------
# 6. Сохранение обучающих пар
# --------------------------------------------------

with (ARTIFACTS / "preferences.jsonl").open(
    "w", encoding="utf-8"
) as f:
    for p in train_preferences:
        f.write(
            json.dumps(p, ensure_ascii=False) + "\n"
        )

# --------------------------------------------------
# 7. Сохранение всех тестовых manifest
# --------------------------------------------------

for split, test_records in test_splits.items():

    save_json(
        f"{split}_manifest.json",
        test_records
    )

# --------------------------------------------------
# 8. Сохранение аудита
# --------------------------------------------------

save_json("pair_audit.json", pair_audit)

# --------------------------------------------------
# 9. Итоговая статистика
# --------------------------------------------------

print("\n========== DATASET AUDIT ==========")

print("Train:", len(train_records))
print("Preference pairs:", len(train_preferences))

for split, test_records in test_splits.items():

    audit = overlap_audit[split]

    print(f"\n{split}:")
    print("  Samples:", len(test_records))
    print("  ID overlap:", audit["id_overlap_count"])
    print(
        "  Image overlap:",
        audit["image_hash_overlap_count"]
    )

print("\n========== LENGTH ANALYSIS ==========")
print("Chosen longer:", chosen_longer)
print("Rejected longer:", rejected_longer)
print("Equal length:", equal_length)

print("\nВсе файлы сохранены.")


Проверка пересечений пройдена для всех тестовых выборок.

========== DATASET AUDIT ==========
Train: 512
Preference pairs: 512

validation:
  Samples: 64
  ID overlap: 0
  Image overlap: 0

id_test:
  Samples: 64
  ID overlap: 0
  Image overlap: 0

ood_test:
  Samples: 64
  ID overlap: 0
  Image overlap: 0

general_test:
  Samples: 64
  ID overlap: 0
  Image overlap: 0

========== LENGTH ANALYSIS ==========
Chosen longer: 0
Rejected longer: 0
Equal length: 512

Все файлы сохранены.


### Начальная и фиксированная политики

*Вес: 1 балл.*

**Постановка.** Создайте обучаемый адаптер и отдельный фиксированный reference. Используйте адаптер работы 09, если он доступен; иначе обе политики начинаются с опубликованной instruction-tuned модели. Запишите выбранный источник.

**Результат.** reference_policy.json.

**Критерий принятия.** Начальные политики совпадают; параметры reference не обновляются; запуск без адаптера 09 не называется продолжением собственного SFT.


In [9]:
!pip install -U bitsandbytes>=0.46.1

In [6]:
from peft import prepare_model_for_kbit_training
processor = AutoProcessor.from_pretrained(MODEL_ID)

adapter_available = (
    (ADAPTER_DIR / "adapter_config.json").is_file()
    and (
        (ADAPTER_DIR / "adapter_model.safetensors").is_file()
        or (ADAPTER_DIR / "adapter_model.bin").is_file()
    )
)

# Для экономии VRAM используем 4-bit загрузку на GPU.
load_kwargs = {
    "torch_dtype": DTYPE,
    "device_map": "auto"
}

use_4bit = torch.cuda.is_available()

if use_4bit:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True
    )

if adapter_available:
    adapter_config = PeftConfig.from_pretrained(ADAPTER_DIR)

    assert (
        adapter_config.base_model_name_or_path.rstrip("/")
        == MODEL_ID.rstrip("/")
    ), (
        "Базовая модель адаптера не совпадает с MODEL_ID. "
        "Проверьте adapter_config.json."
    )

    # Загружаем две независимые копии базовой модели.
    policy_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, **load_kwargs
    )

    reference_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, **load_kwargs
    )

    # Подготавливаем policy для обучения LoRA на 4-bit модели.
    if use_4bit:
        policy_base = prepare_model_for_kbit_training(
            policy_base
        )

    # Один и тот же обученный SFT-адаптер.
    policy_model = PeftModel.from_pretrained(
        policy_base,
        ADAPTER_DIR,
        is_trainable=True
    )

    reference_model = PeftModel.from_pretrained(
        reference_base,
        ADAPTER_DIR,
        is_trainable=False
    )

    source = "lab9_sft_adapter"

else:
    # Fallback: обе политики начинаются с одной instruction-tuned модели.
    policy_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, **load_kwargs
    )

    reference_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, **load_kwargs
    )

    if use_4bit:
        policy_base = prepare_model_for_kbit_training(
            policy_base
        )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]
    )

    policy_model = get_peft_model(
        policy_base, lora_config
    )

    reference_model = get_peft_model(
        reference_base, lora_config
    )

    source = "base_instruction_tuned_no_lab9_adapter"

# Reference не обновляется.
reference_model.eval()

for parameter in reference_model.parameters():
    parameter.requires_grad = False

# Policy остаётся обучаемой.
policy_model.train()

# Включаем gradient checkpointing для экономии видеопамяти.
policy_model.gradient_checkpointing_enable()
policy_model.enable_input_require_grads()

# Проверяем, что обучаются только параметры policy.
trainable_params = sum(
    p.numel()
    for p in policy_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in policy_model.parameters()
)

assert trainable_params > 0, "В policy_model нет обучаемых параметров!"

print(f"Обучаемых параметров: {trainable_params:,}")
print(f"Всего параметров: {total_params:,}")
print(f"Доля обучаемых параметров: {trainable_params / total_params:.4%}")

# Сохраняем начальное состояние policy.
INITIAL_ADAPTER_DIR = ARTIFACTS / "dpo_adapter_init"
INITIAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

policy_model.save_pretrained(INITIAL_ADAPTER_DIR)

reference_info = {
    "model_id": MODEL_ID,
    "adapter_source": str(ADAPTER_DIR) if adapter_available else None,
    "initialization_source": source,
    "policy_trainable": True,
    "reference_trainable": False,
    "same_initial_adapter": True,
    "reference_frozen": True,
    "fast_dev_run": FAST_DEV_RUN,
    "dtype": str(DTYPE),
    "quantization": "4-bit NF4" if use_4bit else "none",
    "policy_adapter": str(INITIAL_ADAPTER_DIR),
    "trainable_parameters": trainable_params,
    "total_parameters": total_params
}

save_json("reference_policy.json", reference_info)

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Обучаемых параметров: 14,966,784
Всего параметров: 2,048,991,232
Доля обучаемых параметров: 0.7304%


In [7]:
trainable_params = sum(
    p.numel() for p in policy_model.parameters()
    if p.requires_grad
)

reference_trainable_params = sum(
    p.numel() for p in reference_model.parameters()
    if p.requires_grad
)

print(
    f"Model: {MODEL_ID}\n"
    f"Initialization: {source}\n"
    f"Trainable policy parameters: {trainable_params:,}\n"
    f"Trainable reference parameters: {reference_trainable_params:,}\n"
    f"Reference frozen: {reference_trainable_params == 0}\n"
    f"Adapter saved: {INITIAL_ADAPTER_DIR}\n"
    f"Metadata: {ARTIFACTS / 'reference_policy.json'}"
)

Model: Qwen/Qwen2.5-VL-3B-Instruct
Initialization: lab9_sft_adapter
Trainable policy parameters: 14,966,784
Trainable reference parameters: 0
Reference frozen: True
Adapter saved: artifacts/dpo_adapter_init
Metadata: artifacts/reference_policy.json


### Log-prob мультимодального ответа

*Вес: 1 балл.*

**Постановка.** Реализуйте prepare_answer и completion_logprob. Сохраните все image tensors процессора, исключите prompt из суммы логарифмов вероятностей и учитывайте только токены ответа и окончания.

**Результат.** Функции подготовки и scoring.

**Критерий принятия.** Префикс токенов full совпадает с отдельно токенизированным prompt; выбран правильный сдвиг next-token labels; score дифференцируем по policy.


In [8]:
def prepare_answer(example, answer):
    """
    Формирует мультимодальные входы для вопроса и ответа.
    Возвращает тензоры процессора и длину prompt.
    """

    image = example["image"]

    # Поддерживаем как путь к изображению, так и PIL.Image.
    if isinstance(image, (str, Path)):
        with Image.open(image) as img:
            image = img.convert("RGB")
    else:
        image = image.convert("RGB")

    question = str(example["question"])
    answer = str(answer)

    messages_prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]

    messages_full = messages_prompt + [
        {
            "role": "assistant",
            "content": answer
        }
    ]

    prompt_text = processor.apply_chat_template(
        messages_prompt,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = processor.apply_chat_template(
        messages_full,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt_inputs = processor(
        text=[prompt_text],
        images=[image],
        return_tensors="pt"
    )

    full_inputs = processor(
        text=[full_text],
        images=[image],
        return_tensors="pt"
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]

    # Проверяем совпадение prompt с началом полной последовательности.
    assert torch.equal(
        prompt_inputs["input_ids"][0],
        full_inputs["input_ids"][0, :prompt_len]
    ), (
        "Prompt prefix не совпадает с префиксом full sequence. "
        "Проверьте chat template и формирование сообщений."
    )

    assert full_inputs["input_ids"].shape[1] > prompt_len, (
        "В полной последовательности отсутствуют токены ответа."
    )

    return {
        "model_inputs": dict(full_inputs),
        "prompt_len": prompt_len,
        "answer": answer
    }

def completion_logprob(model, prepared):
    """
    Считает сумму log-probability только токенов ответа.
    Score остаётся дифференцируемым по параметрам policy.
    """

    model_inputs = prepared["model_inputs"]

    # Устройство входного слоя модели.
    device = model.get_input_embeddings().weight.device

    model_inputs = {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in model_inputs.items()
    }

    input_ids = model_inputs["input_ids"]

    attention_mask = model_inputs.get(
        "attention_mask",
        torch.ones_like(input_ids)
    )

    prompt_len = prepared["prompt_len"]

    outputs = model(
        **model_inputs,
        use_cache=False
    )

    # Logits позиции t предсказывают токен позиции t+1.
    logits = outputs.logits[:, :-1, :].float()
    targets = input_ids[:, 1:]

    log_probs = torch.log_softmax(logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=targets.unsqueeze(-1)
    ).squeeze(-1)

    positions = torch.arange(
        targets.shape[1],
        device=device
    )

    # Учитываем только токены после prompt.
    answer_mask = (
        (positions >= prompt_len - 1).unsqueeze(0)
        & attention_mask[:, 1:].bool()
    )

    assert answer_mask.any(), (
        "Не найдено токенов ответа для вычисления log-probability."
    )

    score = (
        token_log_probs * answer_mask
    ).sum()

    return score

# Создаём мультимодальный Dataset.
from datasets import Image as HFImage

preference_dataset = Dataset.from_list(train_preferences)
preference_dataset = preference_dataset.cast_column(
    "image", HFImage()
)

# Проверяем подготовку обоих ответов на одном примере.
sample = preference_dataset[0]

prepared_chosen = prepare_answer(
    sample, sample["chosen"]
)

prepared_rejected = prepare_answer(
    sample, sample["rejected"]
)

# Дифференцируемый score для обучаемой policy.
chosen_score = completion_logprob(
    policy_model, prepared_chosen
)

rejected_score = completion_logprob(
    policy_model, prepared_rejected
)


In [9]:
print(
    f"Question ID: {sample['id']}\n"
    f"Chosen: {sample['chosen']}\n"
    f"Rejected: {sample['rejected']}\n"
    f"Prompt tokens: {prepared_chosen['prompt_len']}\n"
    f"Full tokens: "
    f"{prepared_chosen['model_inputs']['input_ids'].shape[1]}\n"
    f"Chosen log-prob: {chosen_score.item():.4f}\n"
    f"Rejected log-prob: {rejected_score.item():.4f}\n"
    f"Gradient enabled: {chosen_score.requires_grad}"
)

Question ID: scienceqa-train-9
Chosen: A
Rejected: B
Prompt tokens: 545
Full tokens: 548
Chosen log-prob: -0.0061
Rejected log-prob: -5.4424
Gradient enabled: True


### Целевая функция DPO

*Вес: 1 балл.*

**Постановка.** Настройте beta=0.1, AdamW lr=5e-6, accumulation=8 и 40 optimizer steps. Реализуйте отрицательный log-sigmoid разности policy/ref margins; в optimizer включите только trainable adapter.

**Результат.** dpo_protocol.json и optimizer.

**Критерий принятия.** Рост policy-margin при фиксированном reference уменьшает loss; вычисление reference идёт без градиентов.


In [10]:
DPO_BETA = 0.1
DPO_LR = 5e-6
GRAD_ACCUMULATION = 8
DPO_STEPS = 2 if FAST_DEV_RUN else 40

assert DPO_BETA > 0
assert DPO_LR > 0
assert GRAD_ACCUMULATION >= 1
assert DPO_STEPS >= 1

# В optimizer включаем исключительно обучаемые параметры.
trainable_parameters = [
    p for p in policy_model.parameters()
    if p.requires_grad
]

assert trainable_parameters, (
    "Нет обучаемых параметров policy."
)

assert not any(
    p.requires_grad for p in reference_model.parameters()
), "Reference должна быть заморожена."

# Проверяем, что policy и reference не используют общие параметры.
policy_parameter_ids = {
    id(p) for p in policy_model.parameters()
}

reference_parameter_ids = {
    id(p) for p in reference_model.parameters()
}

assert not (
    policy_parameter_ids & reference_parameter_ids
), "Policy и reference неожиданно используют общие параметры."

optimizer = AdamW(
    trainable_parameters,
    lr=DPO_LR
)


def dpo_loss(
    policy_chosen,
    policy_rejected,
    reference_chosen,
    reference_rejected,
    beta=DPO_BETA
):
    """
    Отрицательный log-sigmoid разности policy/reference margins.
    """

    policy_margin = policy_chosen - policy_rejected

    reference_margin = (
        reference_chosen - reference_rejected
    )

    logits = beta * (
        policy_margin - reference_margin
    )

    loss = -F.logsigmoid(logits)

    return loss, policy_margin, reference_margin


# Проверяем математическое свойство функции потерь.
test_margin = torch.tensor(0.5)
test_reference_margin = torch.tensor(0.1)

test_loss_1 = -F.logsigmoid(
    DPO_BETA * (
        test_margin - test_reference_margin
    )
)

test_loss_2 = -F.logsigmoid(
    DPO_BETA * (
        (test_margin + 1.0) - test_reference_margin
    )
)

assert torch.isfinite(test_loss_1)
assert torch.isfinite(test_loss_2)
assert test_loss_2 < test_loss_1

# Сохраняем конфигурацию DPO.
dpo_protocol = {
    "model": MODEL_ID,
    "initialization_source": source,
    "beta": DPO_BETA,
    "optimizer": "AdamW",
    "learning_rate": DPO_LR,
    "gradient_accumulation_steps": GRAD_ACCUMULATION,
    "optimizer_steps": DPO_STEPS,
    "full_budget": 40,
    "smoke_budget": 2,
    "reference_frozen": True,
    "optimizer_parameters": "trainable_adapter_only",
    "loss": (
        "negative_logsigmoid("
        "beta * (policy_margin - reference_margin))"
    ),
    "fast_dev_run": FAST_DEV_RUN,
    "trainable_parameters": sum(
        p.numel() for p in trainable_parameters
    )
}

save_json("dpo_protocol.json", dpo_protocol)

In [11]:
print(
    f"DPO beta: {DPO_BETA}\n"
    f"Optimizer: AdamW\n"
    f"Learning rate: {DPO_LR}\n"
    f"Gradient accumulation: {GRAD_ACCUMULATION}\n"
    f"Optimizer steps: {DPO_STEPS}\n"
    f"Trainable parameters: "
    f"{sum(p.numel() for p in trainable_parameters):,}\n"
    f"Reference frozen: "
    f"{not any(p.requires_grad for p in reference_model.parameters())}\n"
    f"Loss monotonicity test: PASSED\n"
    f"Artifact: {ARTIFACTS / 'dpo_protocol.json'}"
)

DPO beta: 0.1
Optimizer: AdamW
Learning rate: 5e-06
Gradient accumulation: 8
Optimizer steps: 40
Trainable parameters: 14,966,784
Reference frozen: True
Loss monotonicity test: PASSED
Artifact: artifacts/dpo_protocol.json


### Обучение предпочтениям

*Вес: 1 балл.*

**Постановка.** Выполните реальное DPO-обучение, сохраняя loss каждого шага. Не маскируйте NaN. Сохраните обучаемый LoRA в отдельный каталог.

**Результат.** aligned_adapter/train и dpo_metrics.json.

**Критерий принятия.** Обучение выполнило заданный бюджет; loss конечен, адаптер сохранён; smoke не считается полным экспериментом.


In [12]:
ALIGNED_DIR = ARTIFACTS / "aligned_adapter"
ALIGNED_TRAIN_DIR = ALIGNED_DIR / "train"

ALIGNED_TRAIN_DIR.mkdir(parents=True, exist_ok=True)


class DPOTrainer:
    def __init__(
        self,
        policy,
        reference,
        optimizer,
        beta=0.1
    ):
        self.policy = policy
        self.reference = reference
        self.optimizer = optimizer
        self.beta = beta

    def train_step(self, example):
        chosen = prepare_answer(example, example["chosen"])
        rejected = prepare_answer(example, example["rejected"])

        # Policy: с градиентами.
        policy_chosen = completion_logprob(
            self.policy, chosen
        )

        policy_rejected = completion_logprob(
            self.policy, rejected
        )

        # Reference: без градиентов.
        with torch.no_grad():
            reference_chosen = completion_logprob(
                self.reference, chosen
            )

            reference_rejected = completion_logprob(
                self.reference, rejected
            )

        loss, policy_margin, reference_margin = dpo_loss(
            policy_chosen,
            policy_rejected,
            reference_chosen,
            reference_rejected,
            self.beta
        )

        # Для одного preference-примера ожидаем скаляры.
        assert loss.numel() == 1, "DPO loss должен быть скаляром."
        assert policy_margin.numel() == 1
        assert reference_margin.numel() == 1

        return loss, policy_margin, reference_margin


trainer = DPOTrainer(
    policy=policy_model,
    reference=reference_model,
    optimizer=optimizer,
    beta=DPO_BETA
)

training_data = list(train_preferences)
random.shuffle(training_data)

assert training_data, "Нет данных для DPO."

policy_model.train()
reference_model.eval()

optimizer.zero_grad(set_to_none=True)

metrics = []
micro_losses = []
micro_policy_margins = []
micro_reference_margins = []

data_index = 0
start_time = time.time()

for step in range(1, DPO_STEPS + 1):

    for micro_step in range(GRAD_ACCUMULATION):

        example = training_data[data_index % len(training_data)]
        data_index += 1

        loss, policy_margin, reference_margin = (
            trainer.train_step(example)
        )

        if not torch.isfinite(loss).all():
            raise FloatingPointError(
                f"Non-finite DPO loss at step {step}"
            )

        if not torch.isfinite(policy_margin).all():
            raise FloatingPointError(
                f"Non-finite policy margin at step {step}"
            )

        if not torch.isfinite(reference_margin).all():
            raise FloatingPointError(
                f"Non-finite reference margin at step {step}"
            )

        # Накопление градиентов.
        (loss / GRAD_ACCUMULATION).backward()

        micro_losses.append(loss.detach().item())
        micro_policy_margins.append(
            policy_margin.detach().item()
        )
        micro_reference_margins.append(
            reference_margin.detach().item()
        )

    # Проверяем конечность градиентов и ограничиваем их норму.
    grad_norm = torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
        error_if_nonfinite=True
    )

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    step_metrics = {
        "step": step,
        "loss": sum(micro_losses) / len(micro_losses),
        "policy_margin": (
            sum(micro_policy_margins) /
            len(micro_policy_margins)
        ),
        "reference_margin": (
            sum(micro_reference_margins) /
            len(micro_reference_margins)
        ),
        "gradient_norm": float(grad_norm),
        "learning_rate": optimizer.param_groups[0]["lr"]
    }

    if not all(
        math.isfinite(v)
        for v in step_metrics.values()
        if isinstance(v, (int, float))
    ):
        raise FloatingPointError(
            f"Non-finite metrics at step {step}"
        )

    metrics.append(step_metrics)

    micro_losses.clear()
    micro_policy_margins.clear()
    micro_reference_margins.clear()

    save_json("dpo_metrics.json", {
        "fast_dev_run": FAST_DEV_RUN,
        "optimizer_steps_completed": step,
        "metrics": metrics
    })

    if step % 5 == 0 or step == DPO_STEPS:
        checkpoint_dir = (
            ALIGNED_TRAIN_DIR / f"checkpoint-{step}"
        )
        policy_model.save_pretrained(checkpoint_dir)


# Итоговый адаптер.
policy_model.save_pretrained(ALIGNED_TRAIN_DIR)

elapsed = round(time.time() - start_time, 2)

dpo_results = {
    "status": "success",
    "fast_dev_run": FAST_DEV_RUN,
    "full_experiment": not FAST_DEV_RUN,
    "optimizer_steps_completed": len(metrics),
    "requested_optimizer_steps": DPO_STEPS,
    "gradient_accumulation_steps": GRAD_ACCUMULATION,
    "training_examples_processed": data_index,
    "beta": DPO_BETA,
    "learning_rate": DPO_LR,
    "elapsed_seconds": elapsed,
    "final_loss": metrics[-1]["loss"],
    "final_policy_margin": metrics[-1]["policy_margin"],
    "final_reference_margin": metrics[-1]["reference_margin"],
    "adapter_path": str(ALIGNED_TRAIN_DIR)
}

save_json("dpo_metrics.json", {
    **dpo_results,
    "metrics": metrics
})

save_json("dpo_training_summary.json", dpo_results)


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


In [13]:
print(
    f"Training status: success\n"
    f"FAST_DEV_RUN: {FAST_DEV_RUN}\n"
    f"Optimizer steps: {len(metrics)}/{DPO_STEPS}\n"
    f"Processed examples: {data_index}\n"
    f"Initial loss: {metrics[0]['loss']:.4f}\n"
    f"Final loss: {metrics[-1]['loss']:.4f}\n"
    f"Final policy margin: "
    f"{metrics[-1]['policy_margin']:.4f}\n"
    f"Final reference margin: "
    f"{metrics[-1]['reference_margin']:.4f}\n"
    f"Adapter saved: {ALIGNED_TRAIN_DIR}\n"
    f"Metrics: {ARTIFACTS / 'dpo_metrics.json'}"
)

Training status: success
FAST_DEV_RUN: False
Optimizer steps: 40/40
Processed examples: 320
Initial loss: 0.6908
Final loss: 0.6122
Final policy margin: 6.6025
Final reference margin: 4.8906
Adapter saved: artifacts/aligned_adapter/train
Metrics: artifacts/dpo_metrics.json


### Исходные ответы на тесте

*Вес: 1 балл.*

**Постановка.** Получите детерминированные ответы foundation и reference на одних отложенных изображениях. Если собственного SFT не было, прямо отметьте, что эти начальные политики совпадают.

**Результат.** Парные исходные ответы.

**Критерий принятия.** Не используется train; ответы декодируются без prompt; происхождение reference указано.


In [14]:
def generate_answer(model, record):
    """
    Детерминированная генерация ответа только на основе prompt.
    Возвращает исключительно новые токены, без prompt.
    """

    image = Image.open(record["image"]).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": record["question"]}
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt"
    )

    device = model.get_input_embeddings().weight.device

    inputs = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    model.eval()

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            num_beams=1,
            use_cache=True
        )

    answer_tokens = generated[:, input_length:]

    answer = processor.batch_decode(
        answer_tokens,
        skip_special_tokens=True
    )[0].strip()

    return answer

# Проверяем, являются ли Foundation и Reference
# разными начальными моделями.
same_initial_policies = (
    source != "lab9_sft_adapter"
)


def evaluate_test_split(records, split_name):
    """
    Сравнивает Foundation и Reference на одной тестовой выборке.
    """

    assert records, f"{split_name} пуст."

    paired_answers = []

    for record in records:

        # Foundation: базовая модель без LoRA-адаптера.
        with reference_model.disable_adapter():
            foundation_answer = generate_answer(
                reference_model, record
            )

        # Reference: исходная модель с SFT-адаптером.
        reference_answer = generate_answer(
            reference_model, record
        )

        paired_answers.append({
            "id": record["id"],
            "image": record["image"],
            "image_sha256": record["image_sha256"],
            "question": record["question"],
            "ground_truth": record["answer"],
            "foundation_answer": foundation_answer,
            "reference_answer": reference_answer,
            "foundation_source": MODEL_ID,
            "reference_source": source,
            "split": split_name,
            "used_train_data": False
        })

    results = {
        "model": MODEL_ID,
        "split": split_name,
        "samples": len(paired_answers),
        "foundation": "base_instruction_tuned",
        "reference": source,
        "same_initial_policies": same_initial_policies,
        "sft_adapter_available": adapter_available,
        "dpo_training_used": False,
        "generation": {
            "do_sample": False,
            "num_beams": 1,
            "max_new_tokens": 8
        },
        "answers": paired_answers
    }

    return results


# Все три отложенные тестовые выборки.
test_splits = {
    "id_test": split_records["id_test"],
    "ood_test": split_records["ood_test"],
    "general_test": split_records["general_test"]
}

baseline_results_by_split = {}

for split_name, records in test_splits.items():

    print(
        f"Evaluating {split_name}: "
        f"{len(records)} samples"
    )

    split_results = evaluate_test_split(
        records,
        split_name
    )

    baseline_results_by_split[split_name] = split_results

    # Отдельный файл для каждой тестовой выборки.
    save_json(
        f"baseline_answers_{split_name}.json",
        split_results
    )

    print(
        f"Completed {split_name}: "
        f"{split_results['samples']} samples"
    )


# Общий файл со всеми тремя тестовыми выборками.
baseline_summary = {
    "model": MODEL_ID,
    "foundation": "base_instruction_tuned",
    "reference": source,
    "same_initial_policies": same_initial_policies,
    "sft_adapter_available": adapter_available,
    "dpo_training_used": False,
    "splits": {
        name: {
            "samples": result["samples"],
            "file": f"baseline_answers_{name}.json"
        }
        for name, result in baseline_results_by_split.items()
    },
    "total_samples": sum(
        result["samples"]
        for result in baseline_results_by_split.values()
    )
}

save_json(
    "baseline_summary.json",
    baseline_summary
)

Evaluating id_test: 64 samples
Completed id_test: 64 samples
Evaluating ood_test: 64 samples
Completed ood_test: 64 samples
Evaluating general_test: 64 samples
Completed general_test: 64 samples


In [16]:
id_test_results = baseline_results_by_split["id_test"]
paired_answers = id_test_results["answers"]

print(
    f"Test split: ID test\n"
    f"Test examples: {len(paired_answers)}\n"
    f"Foundation model: {MODEL_ID}\n"
    f"Reference source: {source}\n"
    f"SFT adapter available: {adapter_available}\n"
    f"Initial policies coincide: {same_initial_policies}\n"
    f"Foundation answers: "
    f"{len([x for x in paired_answers if x['foundation_answer']])}\n"
    f"Reference answers: "
    f"{len([x for x in paired_answers if x['reference_answer']])}\n"
    f"Training data used: False\n"
    f"Artifact: {ARTIFACTS / 'baseline_answers_id_test.json'}"
)

Test split: ID test
Test examples: 64
Foundation model: Qwen/Qwen2.5-VL-3B-Instruct
Reference source: lab9_sft_adapter
SFT adapter available: True
Initial policies coincide: False
Foundation answers: 64
Reference answers: 64
Training data used: False
Artifact: artifacts/baseline_answers_id_test.json


### Ответы после DPO

*Вес: 1 балл.*

**Постановка.** Активируйте обученный адаптер и получите ответы для тех же ID. Сохраните foundation/reference/aligned вместе с правильной буквой.

**Результат.** blind_predictions.jsonl.

**Критерий принятия.** Каждый ID имеет три фактических ответа; порядок выборок не используется вместо ID.


In [17]:

def generate_prediction(model, record):
    image = Image.open(record["image"]).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": record["question"]}
        ]
    }]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt"
    )

    device = model.get_input_embeddings().weight.device

    inputs = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    model.eval()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            num_beams=1,
            use_cache=True
        )

    generated = output[:, input_length:]

    return processor.batch_decode(
        generated,
        skip_special_tokens=True
    )[0].strip()


# Все три тестовые выборки.
test_splits = {
    "id_test": split_records["id_test"],
    "ood_test": split_records["ood_test"],
    "general_test": split_records["general_test"]
}

# Проверяем, что обученный DPO-адаптер сохранён.
ALIGNED_ADAPTER_DIR = ARTIFACTS / "aligned_adapter" / "train"

assert (ALIGNED_ADAPTER_DIR / "adapter_config.json").exists(), (
    f"DPO adapter not found: {ALIGNED_ADAPTER_DIR}"
)

# Загружаем и активируем обученный DPO-адаптер.
policy_model.load_adapter(
    str(ALIGNED_ADAPTER_DIR),
    adapter_name="aligned"
)

policy_model.set_adapter("aligned")
policy_model.eval()

# Общий список результатов.
all_blind_predictions = []

# Обрабатываем каждую тестовую выборку.
for split_name, records in test_splits.items():

    print(
        f"\nEvaluating {split_name}: "
        f"{len(records)} samples"
    )

    # Загружаем baseline именно для текущей выборки.
    baseline_path = (
        ARTIFACTS / f"baseline_answers_{split_name}.json"
    )

    assert baseline_path.exists(), (
        f"Baseline file not found: {baseline_path}"
    )

    baseline_data = json.loads(
        baseline_path.read_text(encoding="utf-8")
    )

    baseline_by_id = {
        x["id"]: x
        for x in baseline_data["answers"]
    }

    split_predictions = []

    for record in records:

        example_id = record["id"]

        assert example_id in baseline_by_id, (
            f"Нет baseline для {example_id} "
            f"в выборке {split_name}"
        )

        baseline = baseline_by_id[example_id]

        # Foundation и Reference берём из ранее сохранённого baseline.
        foundation_answer = baseline["foundation_answer"]
        reference_answer = baseline["reference_answer"]

        # Aligned: модель с обученным DPO-адаптером.
        aligned_answer = generate_prediction(
            policy_model,
            record
        )

        prediction = {
            "id": example_id,
            "split": split_name,
            "question": record["question"],
            "image": record["image"],
            "correct_answer": record["answer"],
            "foundation_answer": foundation_answer,
            "reference_answer": reference_answer,
            "aligned_answer": aligned_answer
        }

        split_predictions.append(prediction)

    # Сохраняем результаты текущей выборки.
    split_path = (
        ARTIFACTS / f"blind_predictions_{split_name}.jsonl"
    )

    split_path.write_text(
        "\n".join(
            json.dumps(x, ensure_ascii=False)
            for x in split_predictions
        ) + "\n",
        encoding="utf-8"
    )

    all_blind_predictions.extend(split_predictions)

    print(
        f"Completed {split_name}: "
        f"{len(split_predictions)} samples"
    )

# Общий файл со всеми тремя тестовыми выборками.
blind_path = ARTIFACTS / "blind_predictions_all.jsonl"

blind_path.write_text(
    "\n".join(
        json.dumps(x, ensure_ascii=False)
        for x in all_blind_predictions
    ) + "\n",
    encoding="utf-8"
)

print("\nAll test splits completed.")
print(f"Total predictions: {len(all_blind_predictions)}")
print(f"ID test: {len(test_splits['id_test'])}")
print(f"OOD test: {len(test_splits['ood_test'])}")
print(f"General test: {len(test_splits['general_test'])}")
print(f"Combined artifact: {blind_path}")



Evaluating id_test: 64 samples
Completed id_test: 64 samples

Evaluating ood_test: 64 samples
Completed ood_test: 64 samples

Evaluating general_test: 64 samples
Completed general_test: 64 samples

All test splits completed.
Total predictions: 192
ID test: 64
OOD test: 64
General test: 64
Combined artifact: artifacts/blind_predictions_all.jsonl


In [20]:
print(
    f"Test split: ID test\n"
    f"Examples: {len(all_blind_predictions)}\n"
    f"Policies: foundation, reference, aligned\n"
    f"Unique IDs: {len({x['id'] for x in all_blind_predictions})}\n"
    f"Artifact: {blind_path}"
)

Test split: ID test
Examples: 192
Policies: foundation, reference, aligned
Unique IDs: 192
Artifact: artifacts/blind_predictions_all.jsonl


### Автоматическая и ручная оценка

*Вес: 1 балл.*

**Постановка.** Рассчитайте EM по эталонным буквам. Создайте форму ручной проверки предпочтений, галлюцинаций и формата; заполните её самостоятельно после просмотра ответов. Пропуски обозначайте pending, а не нулём.

**Результат.** objective_scores.json и blind_judgements.jsonl.

**Критерий принятия.** EM не назван человеческим win rate; неопределённые суждения не включены в ручные метрики; для финальной сдачи форма заполнена.


In [21]:

import re
from collections import defaultdict


def extract_letter(answer):
    text = str(answer).strip().upper()
    match = re.search(r"\b([A-E])\b", text)
    return match.group(1) if match else None


def is_valid_format(answer):
    # Строгий формат: только одна буква A-E.
    return bool(
        re.fullmatch(r"[A-E]", str(answer).strip().upper())
    )


policies = ["foundation", "reference", "aligned"]

# Используем общий список из предыдущего блока.
predictions = all_blind_predictions

assert predictions, "Список all_blind_predictions пуст."


# Функция расчёта метрик для одной выборки.
def calculate_objective_scores(items):

    scores = {}

    total = len(items)

    for policy in policies:

        correct = 0
        valid = 0

        for item in items:

            predicted = extract_letter(
                item[f"{policy}_answer"]
            )

            target = extract_letter(
                item["correct_answer"]
            )

            if predicted is not None and predicted == target:
                correct += 1

            if is_valid_format(item[f"{policy}_answer"]):
                valid += 1

        scores[policy] = {
            "exact_match": correct / total if total else None,
            "correct": correct,
            "total": total,
            "valid_format": valid / total if total else None,
            "valid_format_count": valid
        }

    return scores


# Считаем метрики отдельно по каждой тестовой выборке.
objective_scores_by_split = {}

for split_name in ["id_test", "ood_test", "general_test"]:

    split_items = [
        item for item in predictions
        if item["split"] == split_name
    ]

    assert split_items, f"Выборка {split_name} пуста."

    objective_scores_by_split[split_name] = (
        calculate_objective_scores(split_items)
    )


# Общие метрики по всем трём выборкам.
objective_scores = calculate_objective_scores(predictions)

# Сохраняем общие и отдельные результаты в одном JSON.
objective_scores["by_split"] = objective_scores_by_split

save_json("objective_scores.json", objective_scores)


# Создаём шаблон ручной оценки для всех тестовых примеров.
# После просмотра ответов замените pending на реальные суждения.
blind_judgements = []

for item in predictions:

    blind_judgements.append({
        "id": item["id"],
        "split": item["split"],
        "preferred": "pending",
        "hallucination": {
            "foundation": "pending",
            "reference": "pending",
            "aligned": "pending"
        },
        "format_ok": {
            "foundation": "pending",
            "reference": "pending",
            "aligned": "pending"
        },
        "notes": ""
    })


# Сохраняем шаблон ручной оценки.
judgements_path = ARTIFACTS / "blind_judgements.jsonl"

judgements_path.write_text(
    "\n".join(
        json.dumps(x, ensure_ascii=False)
        for x in blind_judgements
    ) + "\n",
    encoding="utf-8"
)


# Выводим общие метрики.
print("=" * 60)
print("OBJECTIVE SCORES — ALL TEST SPLITS")
print("=" * 60)

for policy in policies:

    metrics = objective_scores[policy]

    print(
        f"{policy}: "
        f"EM={metrics['correct']}/{metrics['total']} "
        f"({metrics['exact_match']:.3f}), "
        f"valid format="
        f"{metrics['valid_format_count']}/{metrics['total']} "
        f"({metrics['valid_format']:.3f})"
    )


# Выводим метрики отдельно по каждой выборке.
for split_name, split_scores in objective_scores_by_split.items():

    print(f"\n{'=' * 60}")
    print(f"OBJECTIVE SCORES — {split_name.upper()}")
    print("=" * 60)

    for policy in policies:

        metrics = split_scores[policy]

        print(
            f"{policy}: "
            f"EM={metrics['correct']}/{metrics['total']} "
            f"({metrics['exact_match']:.3f}), "
            f"valid format="
            f"{metrics['valid_format_count']}/{metrics['total']} "
            f"({metrics['valid_format']:.3f})"
        )


print("\nManual judgement forms:", len(blind_judgements))
print("Initial status: pending")
print(f"Objective scores: {ARTIFACTS / 'objective_scores.json'}")
print(f"Manual judgements: {judgements_path}")


OBJECTIVE SCORES — ALL TEST SPLITS
foundation: EM=150/192 (0.781), valid format=183/192 (0.953)
reference: EM=157/192 (0.818), valid format=192/192 (1.000)
aligned: EM=156/192 (0.812), valid format=192/192 (1.000)

OBJECTIVE SCORES — ID_TEST
foundation: EM=56/64 (0.875), valid format=57/64 (0.891)
reference: EM=56/64 (0.875), valid format=64/64 (1.000)
aligned: EM=55/64 (0.859), valid format=64/64 (1.000)

OBJECTIVE SCORES — OOD_TEST
foundation: EM=37/64 (0.578), valid format=62/64 (0.969)
reference: EM=44/64 (0.688), valid format=64/64 (1.000)
aligned: EM=44/64 (0.688), valid format=64/64 (1.000)

OBJECTIVE SCORES — GENERAL_TEST
foundation: EM=57/64 (0.891), valid format=64/64 (1.000)
reference: EM=57/64 (0.891), valid format=64/64 (1.000)
aligned: EM=57/64 (0.891), valid format=64/64 (1.000)

Manual judgement forms: 192
Initial status: pending
Objective scores: artifacts/objective_scores.json
Manual judgements: artifacts/blind_judgements.jsonl


In [22]:
import pandas as pd

policies = ["foundation", "reference", "aligned"]
splits = ["id_test", "ood_test", "general_test"]

table_rows = []

for split_name in splits:
    for policy in policies:

        metrics = objective_scores_by_split[split_name][policy]

        table_rows.append({
            "Test split": split_name,
            "Model": policy.capitalize(),
            "Exact Match (%)": (
                metrics["exact_match"] * 100
                if metrics["exact_match"] is not None else None
            ),
            "Valid Format (%)": (
                metrics["valid_format"] * 100
                if metrics["valid_format"] is not None else None
            ),
            "Correct": metrics["correct"],
            "Total": metrics["total"]
        })

# Общая таблица.
overall_rows = []

for policy in policies:
    metrics = objective_scores[policy]

    overall_rows.append({
        "Test split": "ALL",
        "Model": policy.capitalize(),
        "Exact Match (%)": metrics["exact_match"] * 100,
        "Valid Format (%)": metrics["valid_format"] * 100,
        "Correct": metrics["correct"],
        "Total": metrics["total"]
    })

results_table = pd.DataFrame(table_rows + overall_rows)

# Округление процентов до двух знаков.
results_table["Exact Match (%)"] = (
    results_table["Exact Match (%)"].round(2)
)
results_table["Valid Format (%)"] = (
    results_table["Valid Format (%)"].round(2)
)

display(results_table)

print(
    f"\nObjective evaluation: {len(all_blind_predictions)} examples"
)
print(f"Policies evaluated: {len(policies)}")
print(f"Manual judgements: {len(blind_judgements)}")
print(f"Completed judgements: 0")
print(f"Pending judgements: {len(blind_judgements)}")

,Test split,Model,Exact Match (%),Valid Format (%),Correct,Total
0,id_test,Foundation,87.50,89.06,56,64
1,id_test,Reference,87.50,100.00,56,64
2,id_test,Aligned,85.94,100.00,55,64
3,ood_test,Foundation,57.81,96.88,37,64
4,ood_test,Reference,68.75,100.00,44,64
5,ood_test,Aligned,68.75,100.00,44,64
6,general_test,Foundation,89.06,100.00,57,64
7,general_test,Reference,89.06,100.00,57,64
8,general_test,Aligned,89.06,100.00,57,64
9,ALL,Foundation,78.12,95.31,150,192



Objective evaluation: 192 examples
Policies evaluated: 3
Manual judgements: 192
Completed judgements: 0
Pending judgements: 192


### Эффект DPO

*Вес: 1 балл.*

**Постановка.** Сопоставьте EM трёх политик, а по заполненным суждениям — win rate, hallucination rate и соблюдение формата. Укажите число оценённых записей и ограничения label-derived предпочтений.

**Результат.** blind_report.json и вывод.

**Критерий принятия.** Метрики имеют реальные знаменатели; pending-оценка не выдаётся за завершённую; вывод не обобщает предпочтение правильной буквы на безопасность или универсальную полезность.


In [23]:
# Читаем ручные оценки.
judgements = [
    json.loads(line)
    for line in judgements_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

predictions = all_blind_predictions

assert len({x["id"] for x in judgements}) == len(judgements), \
    "В ручных оценках есть повторяющиеся ID"

prediction_ids = {x["id"] for x in predictions}
judgement_ids = {x["id"] for x in judgements}

assert prediction_ids == judgement_ids, \
    "ID ручных оценок не совпадают с ID ответов"

# Проверяем соответствие ID внутри каждой выборки.
for split_name in ["id_test", "ood_test", "general_test"]:
    split_prediction_ids = {
        x["id"] for x in predictions
        if x["split"] == split_name
    }
    split_judgement_ids = {
        x["id"] for x in judgements
        if x["split"] == split_name
    }

    assert split_prediction_ids == split_judgement_ids, \
        f"ID ручных оценок не совпадают для {split_name}"

# Считаем только заполненные оценки предпочтений.
completed_preferences = [
    x for x in judgements
    if x.get("preferred") in policies + ["tie"]
]

decisive_preferences = [
    x for x in completed_preferences
    if x["preferred"] != "tie"
]

wins = {
    policy: sum(
        x["preferred"] == policy
        for x in decisive_preferences
    )
    for policy in policies
}

denominator = len(decisive_preferences)

win_rates = {
    policy: wins[policy] / denominator if denominator else None
    for policy in policies
}


def manual_rate(items, field, policy):
    values = [
        x[field][policy]
        for x in items
        if isinstance(x.get(field), dict)
        and x[field].get(policy) in [True, False]
    ]

    if not values:
        return {
            "rate": None,
            "count": 0
        }

    return {
        "rate": sum(values) / len(values),
        "count": len(values)
    }


def calculate_human_evaluation(items):
    completed = [
        x for x in items
        if x.get("preferred") in policies + ["tie"]
    ]

    decisive = [
        x for x in completed
        if x["preferred"] != "tie"
    ]

    local_wins = {
        policy: sum(
            x["preferred"] == policy
            for x in decisive
        )
        for policy in policies
    }

    local_denominator = len(decisive)

    local_win_rates = {
        policy: (
            local_wins[policy] / local_denominator
            if local_denominator else None
        )
        for policy in policies
    }

    return {
        "total_records": len(items),
        "completed_preferences": len(completed),
        "decisive_preferences": local_denominator,
        "ties": len(completed) - local_denominator,
        "pending_preferences": len(items) - len(completed),
        "wins": local_wins,
        "win_rates": local_win_rates,
        "hallucination_rates": {
            policy: manual_rate(items, "hallucination", policy)
            for policy in policies
        },
        "format_rates": {
            policy: manual_rate(items, "format_ok", policy)
            for policy in policies
        }
    }


def calculate_split_report(split_name, split_predictions, split_judgements):
    # Объективные метрики берём из ранее рассчитанных результатов.
    objective = (
        objective_scores_by_split[split_name]
        if split_name != "ALL"
        else objective_scores
    )

    return {
        "examples": len(split_predictions),
        "objective_em": {
            policy: objective[policy]["exact_match"]
            for policy in policies
        },
        "objective_correct_counts": {
            policy: {
                "correct": objective[policy]["correct"],
                "total": objective[policy]["total"]
            }
            for policy in policies
        },
        "human_evaluation": calculate_human_evaluation(split_judgements)
    }


# Формируем отчёты по каждой выборке.
report_by_split = {}

for split_name in ["id_test", "ood_test", "general_test"]:
    split_predictions = [
        x for x in predictions
        if x["split"] == split_name
    ]

    split_judgements = [
        x for x in judgements
        if x["split"] == split_name
    ]

    report_by_split[split_name] = calculate_split_report(
        split_name,
        split_predictions,
        split_judgements
    )

# Общий отчёт по всем тестовым выборкам.
report_by_split["ALL"] = calculate_split_report(
    "ALL",
    predictions,
    judgements
)

report = {
    "test_splits": report_by_split,
    "limitations": [
        "Preference pairs were derived from dataset labels, not human comparisons.",
        "Choosing the correct answer letter does not establish safety or universal helpfulness.",
        "EM measures exact-match answer accuracy, not human preference.",
        "Pending judgements are excluded from manual metrics.",
        "Win rates are calculated only from decisive preferences; ties are excluded.",
        "Human evaluation metrics depend on the number of manually completed judgements."
    ]
}

save_json("blind_report.json", report)

print("Blind evaluation report saved.")
print(f"Total predictions: {len(predictions)}")
print(f"Completed preferences: {len(completed_preferences)}")
print(f"Decisive preferences: {denominator}")
print(f"Pending preferences: {len(judgements) - len(completed_preferences)}")

Blind evaluation report saved.
Total predictions: 192
Completed preferences: 0
Decisive preferences: 0
Pending preferences: 192


In [29]:
import json
from collections import Counter

# Перечитываем файл с диска
judgements = [
    json.loads(line)
    for line in judgements_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Файл:", judgements_path.resolve())
print("Всего записей:", len(judgements))

for split_name in ["id_test", "ood_test", "general_test"]:
    items = [x for x in judgements if x["split"] == split_name]
    counts = Counter(x.get("preferred", "pending") for x in items)

    completed = sum(counts[p] for p in policies + ["tie"])
    pending = counts.get("pending", 0)

    print(f"\n{split_name}:")
    print(f"Всего: {len(items)}")
    print(f"Заполнено: {completed}")
    print(f"Ожидают: {pending}")
    print(f"Распределение: {dict(counts)}")

Файл: /content/artifacts/blind_judgements.jsonl
Всего записей: 192

id_test:
Всего: 64
Заполнено: 64
Ожидают: 0
Распределение: {'tie': 51, 'foundation': 6, 'reference': 7}

ood_test:
Всего: 64
Заполнено: 64
Ожидают: 0
Распределение: {'foundation': 6, 'reference': 22, 'tie': 36}

general_test:
Всего: 64
Заполнено: 64
Ожидают: 0
Распределение: {'tie': 57, 'reference': 6, 'foundation': 1}


In [32]:
import json
import pandas as pd
from collections import Counter
from IPython.display import display, Markdown

# ============================================================
# ИТОГОВЫЙ ОТЧЁТ ПО ОБЪЕКТИВНОЙ И РУЧНОЙ ОЦЕНКЕ
# ============================================================

display(Markdown("# Итоговая оценка моделей"))

# ------------------------------------------------------------
# 0. Проверка исходных данных и перечитывание ручной разметки
# ------------------------------------------------------------

policies = ["foundation", "reference", "aligned"]
split_names = ["id_test", "ood_test", "general_test"]

assert ARTIFACTS.exists(), f"Папка ARTIFACTS не найдена: {ARTIFACTS}"
assert judgements_path.exists(), (
    f"Файл ручной разметки не найден: {judgements_path}"
)
assert all_blind_predictions, "Список all_blind_predictions пуст."

# ВАЖНО: загружаем актуальную разметку с диска,
# а не используем старую переменную judgements.
judgements = [
    json.loads(line)
    for line in judgements_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

predictions = all_blind_predictions

# Проверяем уникальность ID.
judgement_ids = [x["id"] for x in judgements]
prediction_ids = [x["id"] for x in predictions]

assert len(judgement_ids) == len(set(judgement_ids)), (
    "В файле ручной разметки есть повторяющиеся ID."
)
assert len(prediction_ids) == len(set(prediction_ids)), (
    "В all_blind_predictions есть повторяющиеся ID."
)
assert set(judgement_ids) == set(prediction_ids), (
    "ID в ручной разметке не совпадают с ID предсказаний."
)

# Проверяем соответствие split.
prediction_split_by_id = {x["id"]: x["split"] for x in predictions}

for item in judgements:
    assert item["split"] == prediction_split_by_id[item["id"]], (
        f"Несовпадение split для {item['id']}"
    )

print(f"Файл разметки: {judgements_path.resolve()}")
print(f"Всего предсказаний: {len(predictions)}")
print(f"Всего записей разметки: {len(judgements)}")

# ------------------------------------------------------------
# 1. Пересчёт ручной оценки
# ------------------------------------------------------------

def manual_rate(items, field, policy):
    values = [
        item[field][policy]
        for item in items
        if isinstance(item.get(field, {}).get(policy), bool)
    ]

    if not values:
        return {"rate": None, "count": 0}

    return {
        "rate": sum(values) / len(values),
        "count": len(values)
    }


def calculate_human_evaluation(items):
    total = len(items)

    completed_preferences = [
        x for x in items
        if x.get("preferred") in policies + ["tie"]
    ]

    decisive_preferences = [
        x for x in completed_preferences
        if x["preferred"] != "tie"
    ]

    ties = sum(
        x["preferred"] == "tie"
        for x in completed_preferences
    )

    wins = {
        policy: sum(
            x["preferred"] == policy
            for x in decisive_preferences
        )
        for policy in policies
    }

    decisive_count = len(decisive_preferences)

    win_rates = {
        policy: (
            wins[policy] / decisive_count
            if decisive_count else None
        )
        for policy in policies
    }

    hallucination_rates = {
        policy: manual_rate(items, "hallucination", policy)
        for policy in policies
    }

    format_rates = {
        policy: manual_rate(items, "format_ok", policy)
        for policy in policies
    }

    return {
        "total_records": total,
        "completed_preferences": len(completed_preferences),
        "decisive_preferences": decisive_count,
        "ties": ties,
        "pending_preferences": total - len(completed_preferences),
        "wins": wins,
        "win_rates": win_rates,
        "hallucination_rates": hallucination_rates,
        "format_rates": format_rates
    }


def calculate_split_report(split_items, split_judgements):
    objective = calculate_objective_scores(split_items)
    human = calculate_human_evaluation(split_judgements)

    return {
        "objective_em": {
            policy: objective[policy]["exact_match"]
            for policy in policies
        },
        "objective_correct_counts": {
            policy: {
                "correct": objective[policy]["correct"],
                "total": objective[policy]["total"]
            }
            for policy in policies
        },
        "human_evaluation": human
    }


# ------------------------------------------------------------
# 2. Формирование нового report
# ------------------------------------------------------------

report = {"test_splits": {}, "limitations": []}

for split_name in split_names:
    split_predictions = [
        x for x in predictions
        if x["split"] == split_name
    ]

    split_judgements = [
        x for x in judgements
        if x["split"] == split_name
    ]

    assert split_predictions, f"Нет предсказаний для {split_name}"
    assert split_judgements, f"Нет разметки для {split_name}"

    report["test_splits"][split_name] = calculate_split_report(
        split_predictions,
        split_judgements
    )

# Общий отчёт по всем выборкам.
report["test_splits"]["ALL"] = calculate_split_report(
    predictions,
    judgements
)

report["limitations"] = [
    "Ручная оценка зависит от субъективности разметчика.",
    "Неразмеченные значения не учитываются при расчёте соответствующих показателей.",
    "Доля побед рассчитывается только по решающим предпочтениям; ничьи исключены."
]

# Сохраняем обновлённый JSON-отчёт.
save_json("blind_report.json", report)

# ------------------------------------------------------------
# 3. Объективная точность
# ------------------------------------------------------------

objective_rows = []

for split_name in split_names + ["ALL"]:
    if split_name == "ALL":
        split_metrics = calculate_objective_scores(predictions)
    else:
        split_items = [
            x for x in predictions
            if x["split"] == split_name
        ]
        split_metrics = calculate_objective_scores(split_items)

    for policy in policies:
        m = split_metrics[policy]

        objective_rows.append({
            "Выборка": split_name,
            "Модель": policy,
            "Правильных ответов": f"{m['correct']}/{m['total']}",
            "Exact Match, %": round(m["exact_match"] * 100, 2),
            "Корректный формат, %": round(
                m["valid_format"] * 100, 2
            )
        })

objective_table = pd.DataFrame(objective_rows)

display(Markdown("## 1. Объективные метрики"))
display(objective_table)

# ------------------------------------------------------------
# 4. Ручная оценка: предпочтения
# ------------------------------------------------------------

human_rows = []

for split_name, split_report in report["test_splits"].items():
    human = split_report["human_evaluation"]

    for policy in policies:
        rate = human["win_rates"][policy]

        human_rows.append({
            "Выборка": split_name,
            "Модель": policy,
            "Победы": human["wins"][policy],
            "Доля побед, %": (
                round(rate * 100, 2)
                if rate is not None else "—"
            ),
            "Всего решающих сравнений": human["decisive_preferences"]
        })

human_preferences_table = pd.DataFrame(human_rows)

display(Markdown("## 2. Ручная оценка: предпочтения"))
display(human_preferences_table)

# ------------------------------------------------------------
# 5. Ручная оценка: галлюцинации и формат
# ------------------------------------------------------------

quality_rows = []

for split_name, split_report in report["test_splits"].items():
    human = split_report["human_evaluation"]

    for policy in policies:
        hallucination = human["hallucination_rates"][policy]
        format_result = human["format_rates"][policy]

        quality_rows.append({
            "Выборка": split_name,
            "Модель": policy,
            "Галлюцинации, %": (
                round(hallucination["rate"] * 100, 2)
                if hallucination["rate"] is not None else "—"
            ),
            "Оценено на галлюцинации": hallucination["count"],
            "Корректный формат, %": (
                round(format_result["rate"] * 100, 2)
                if format_result["rate"] is not None else "—"
            ),
            "Оценено на формат": format_result["count"]
        })

quality_table = pd.DataFrame(quality_rows)

display(Markdown("## 3. Ручная оценка: качество ответов"))
display(quality_table)

# ------------------------------------------------------------
# 6. Полнота ручной разметки
# ------------------------------------------------------------

completion_rows = []

for split_name, split_report in report["test_splits"].items():
    human = split_report["human_evaluation"]

    completion_rows.append({
        "Выборка": split_name,
        "Всего примеров": human["total_records"],
        "Заполнено предпочтений": human["completed_preferences"],
        "Решающих сравнений": human["decisive_preferences"],
        "Ничьих": human["ties"],
        "Ожидают разметки": human["pending_preferences"]
    })

completion_table = pd.DataFrame(completion_rows)

display(Markdown("## 4. Полнота ручной разметки"))
display(completion_table)

# ------------------------------------------------------------
# 7. Диагностика разметки по моделям
# ------------------------------------------------------------

display(Markdown("## 5. Диагностика ручной разметки"))

for split_name in split_names:
    items = [x for x in judgements if x["split"] == split_name]
    counts = Counter(x.get("preferred", "pending") for x in items)

    print(f"\n{split_name}:")
    print(f"Всего примеров: {len(items)}")
    print(
        "Заполнено предпочтений:",
        sum(counts[p] for p in policies + ["tie"])
    )
    print("Ожидают предпочтения:", counts.get("pending", 0))
    print("Распределение:", dict(counts))

    for policy in policies:
        hallucination_count = sum(
            isinstance(x.get("hallucination", {}).get(policy), bool)
            for x in items
        )
        format_count = sum(
            isinstance(x.get("format_ok", {}).get(policy), bool)
            for x in items
        )

        print(
            f"  {policy}: оценено на галлюцинации "
            f"{hallucination_count}/{len(items)}, "
            f"оценено на формат {format_count}/{len(items)}"
        )

# ------------------------------------------------------------
# 8. Краткий итог
# ------------------------------------------------------------

all_human = report["test_splits"]["ALL"]["human_evaluation"]

display(Markdown("## 6. Итог"))

print(f"Всего тестовых примеров: {len(predictions)}")
print(
    f"Заполнено предпочтений: "
    f"{all_human['completed_preferences']}/"
    f"{all_human['total_records']}"
)
print(f"Решающие сравнения: {all_human['decisive_preferences']}")
print(f"Ничьи: {all_human['ties']}")
print(f"Ожидают разметки: {all_human['pending_preferences']}")

best_policy = max(
    policies,
    key=lambda p: objective_scores[p]["exact_match"]
)

print(
    f"\nЛучшая модель по Exact Match: {best_policy} "
    f"({objective_scores[best_policy]['exact_match'] * 100:.2f}%)"
)

available_win_rates = {
    p: all_human["win_rates"][p]
    for p in policies
    if all_human["win_rates"][p] is not None
}

if available_win_rates:
    best_human_policy = max(
        available_win_rates,
        key=available_win_rates.get
    )

    print(
        f"Лучшая модель по ручным предпочтениям: "
        f"{best_human_policy} "
        f"({available_win_rates[best_human_policy] * 100:.2f}%)"
    )
else:
    print("Ручных сравнений пока недостаточно для определения лидера.")

# ------------------------------------------------------------
# 9. Сохранение таблиц
# ------------------------------------------------------------

objective_table.to_csv(
    ARTIFACTS / "final_objective_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

human_preferences_table.to_csv(
    ARTIFACTS / "final_human_preferences.csv",
    index=False,
    encoding="utf-8-sig"
)

quality_table.to_csv(
    ARTIFACTS / "final_human_quality.csv",
    index=False,
    encoding="utf-8-sig"
)

completion_table.to_csv(
    ARTIFACTS / "final_annotation_completion.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\nВсе таблицы сохранены в: {ARTIFACTS}")
print(f"Полный JSON-отчёт: {ARTIFACTS / 'blind_report.json'}")

# Итоговая оценка моделей

Файл разметки: /content/artifacts/blind_judgements.jsonl
Всего предсказаний: 192
Всего записей разметки: 192


## 1. Объективные метрики

,Выборка,Модель,Правильных ответов,"Exact Match, %","Корректный формат, %"
0,id_test,foundation,56/64,87.50,89.06
1,id_test,reference,56/64,87.50,100.00
2,id_test,aligned,55/64,85.94,100.00
3,ood_test,foundation,37/64,57.81,96.88
4,ood_test,reference,44/64,68.75,100.00
5,ood_test,aligned,44/64,68.75,100.00
6,general_test,foundation,57/64,89.06,100.00
7,general_test,reference,57/64,89.06,100.00
8,general_test,aligned,57/64,89.06,100.00
9,ALL,foundation,150/192,78.12,95.31


## 2. Ручная оценка: предпочтения

...

,Выборка,Всего примеров,Заполнено предпочтений,Решающих сравнений,Ничьих,Ожидают разметки
0,id_test,64,64,13,51,0
1,ood_test,64,64,28,36,0
2,general_test,64,64,7,57,0
3,ALL,192,192,48,144,0


## 5. Диагностика ручной разметки


id_test:
Всего примеров: 64
Заполнено предпочтений: 64
Ожидают предпочтения: 0
Распределение: {'tie': 51, 'foundation': 6, 'reference': 7}
  foundation: оценено на галлюцинации 64/64, оценено на формат 64/64
  reference: оценено на галлюцинации 64/64, оценено на формат 64/64
  aligned: оценено на галлюцинации 64/64, оценено на формат 64/64

ood_test:
Всего примеров: 64
Заполнено предпочтений: 64
Ожидают предпочтения: 0
Распределение: {'foundation': 6, 'reference': 22, 'tie': 36}
  foundation: оценено на галлюцинации 64/64, оценено на формат 64/64
  reference: оценено на галлюцинации 64/64, оценено на формат 64/64
  aligned: оценено на галлюцинации 64/64, оценено на формат 64/64

general_test:
Всего примеров: 64
Заполнено предпочтений: 64
Ожидают предпочтения: 0
Распределение: {'tie': 57, 'reference': 6, 'foundation': 1}
  foundation: оценено на галлюцинации 64/64, оценено на формат 64/64
  reference: оценено на галлюцинации 64/64, оценено на формат 64/64
  aligned: оценено на галлюцин

## 6. Итог

Всего тестовых примеров: 192
Заполнено предпочтений: 192/192
Решающие сравнения: 48
Ничьи: 144
Ожидают разметки: 0

Лучшая модель по Exact Match: reference (81.77%)
Лучшая модель по ручным предпочтениям: reference (72.92%)

Все таблицы сохранены в: /content/artifacts
Полный JSON-отчёт: /content/artifacts/blind_report.json


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [24]:
import re
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

ARTIFACTS = Path("artifacts").resolve()
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# ============================================================
# Бонусное исследование:
# влияет ли тип изображения на эффект SFT и DPO?
# ============================================================

# ID рисунков и схем
drawing_ids = {
    10, 104, 107, 110, 136, 148, 154, 157, 167, 17, 172,
    180, 194, 236, 243, 247, 249, 253, 257, 263, 80, 94
}

# ID фотографий
photo_ids = {
    116, 123, 150, 161, 168, 177, 18, 185, 207, 219, 241,
    256, 272, 349, 36, 382, 397, 398, 400, 402, 49, 78
}

assert not (drawing_ids & photo_ids), \
    "ID рисунков и фотографий пересекаются!"

assert len(drawing_ids) == 22 and len(photo_ids) == 22, \
    "Ожидалось по 22 ID в каждой группе."

# ============================================================
# Вспомогательные функции
# ============================================================

def normalize_id(value):
    """scienceqa-test-104 -> 104"""
    match = re.search(r"(\d+)$", str(value))
    return int(match.group(1)) if match else None


def extract_answer(value):
    """Извлекает букву правильного варианта A-E."""
    text = str(value).strip().upper()
    match = re.search(r"\b([A-E])\b", text)
    return match.group(1) if match else None


# ============================================================
# Подготовка предсказаний
# ============================================================

# Используем ранее полученные предсказания трёх политик.
# all_blind_predictions должен быть создан в предыдущих ячейках.

assert "all_blind_predictions" in globals(), \
    "Сначала выполни блок генерации all_blind_predictions."

predictions = all_blind_predictions

rows = []
found_ids = set()

for p in predictions:
    if p.get("split") != "id_test":
        continue

    example_id = normalize_id(p.get("id"))

    if example_id in drawing_ids:
        group = "Рисунки"
    elif example_id in photo_ids:
        group = "Фотографии"
    else:
        continue

    found_ids.add(example_id)

    rows.append({
        "id": p["id"],
        "Тип изображения": group,
        "Правильный ответ": extract_answer(p["correct_answer"]),
        "Foundation": extract_answer(p["foundation_answer"]),
        "SFT": extract_answer(p["reference_answer"]),
        "DPO": extract_answer(p["aligned_answer"])
    })

details = pd.DataFrame(rows)

if details.empty:
    raise ValueError(
        "Не найдено ни одного подходящего примера. "
        "Проверь all_blind_predictions и ID."
    )

# Проверяем полноту набора.
expected_ids = drawing_ids | photo_ids
missing_ids = sorted(expected_ids - found_ids)

print(f"Найдено примеров: {len(found_ids)} из {len(expected_ids)}")

if missing_ids:
    print("Не найдены ID:", missing_ids)

# ============================================================
# Подсчёт точности
# ============================================================

for policy in ["Foundation", "SFT", "DPO"]:
    details[f"{policy} верно"] = (
        details[policy] == details["Правильный ответ"]
    ) & details["Правильный ответ"].notna()


summary_rows = []

for group in ["Фотографии", "Рисунки"]:
    part = details[details["Тип изображения"] == group]

    if part.empty:
        continue

    n = len(part)

    foundation_acc = part["Foundation верно"].mean() * 100
    sft_acc = part["SFT верно"].mean() * 100
    dpo_acc = part["DPO верно"].mean() * 100

    summary_rows.append({
        "Тип изображения": group,
        "Примеров": n,
        "Foundation, %": round(foundation_acc, 2),
        "SFT, %": round(sft_acc, 2),
        "DPO, %": round(dpo_acc, 2),
        "Прирост SFT, п.п.": round(sft_acc - foundation_acc, 2),
        "Прирост DPO, п.п.": round(dpo_acc - sft_acc, 2),
        "Общий прирост, п.п.": round(dpo_acc - foundation_acc, 2)
    })

summary = pd.DataFrame(summary_rows)

# ============================================================
# Сравнение эффектов на фотографиях и рисунках
# ============================================================

summary_by_group = summary.set_index("Тип изображения")

required_groups = {"Фотографии", "Рисунки"}

if required_groups.issubset(summary_by_group.index):
    photo = summary_by_group.loc["Фотографии"]
    drawing = summary_by_group.loc["Рисунки"]

    sft_difference = (
        photo["Прирост SFT, п.п."]
        - drawing["Прирост SFT, п.п."]
    )

    dpo_difference = (
        photo["Прирост DPO, п.п."]
        - drawing["Прирост DPO, п.п."]
    )

    total_difference = (
        photo["Общий прирост, п.п."]
        - drawing["Общий прирост, п.п."]
    )

    if total_difference > 0:
        conclusion = (
            "Общий эффект SFT и DPO оказался выше на фотографиях. "
            "Результаты согласуются с гипотезой."
        )
    elif total_difference < 0:
        conclusion = (
            "Общий эффект SFT и DPO оказался выше на рисунках. "
            "Результаты не поддерживают гипотезу."
        )
    else:
        conclusion = (
            "Общий прирост точности одинаков для фотографий и рисунков. "
            "Гипотеза не получила явной поддержки."
        )

else:
    sft_difference = None
    dpo_difference = None
    total_difference = None

    conclusion = (
        "Для сравнения групп недостаточно данных: "
        "одна из групп не содержит найденных примеров."
    )

# ============================================================
# Визуализация результатов
# ============================================================

display(Markdown("## Бонусное исследование: фотографии и рисунки"))

display(Markdown(
    "**Гипотеза:** эффект дообучения SFT и последующего DPO "
    "на точность ответов зависит от типа изображения. "
    "Предполагается, что общий прирост точности на фотографиях "
    "будет выше, чем на рисунках."
))

display(Markdown("### Сравнение точности"))

display(summary)

display(Markdown("### Предсказания по отдельным примерам"))

display(details)

if sft_difference is not None:
    comparison = pd.DataFrame([
        {
            "Этап": "SFT относительно Foundation",
            "Разница прироста (фото − рисунки), п.п.": round(
                sft_difference, 2
            )
        },
        {
            "Этап": "DPO относительно SFT",
            "Разница прироста (фото − рисунки), п.п.": round(
                dpo_difference, 2
            )
        },
        {
            "Этап": "DPO относительно Foundation",
            "Разница прироста (фото − рисунки), п.п.": round(
                total_difference, 2
            )
        }
    ])

    display(Markdown("### Различия между группами"))
    display(comparison)

# ============================================================
# Сохранение результатов
# ============================================================

details_path = ARTIFACTS / "bonus_photo_drawing_details.csv"
summary_path = ARTIFACTS / "bonus_photo_drawing_summary.csv"

details.to_csv(
    details_path,
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# Формирование bonus.md
# ============================================================

def format_value(value):
    if value is None:
        return "недостаточно данных"
    return f"{value:.2f}"

if sft_difference is not None:
    comparison_text = f"""
| Сравнение | Разница прироста (фото − рисунки) |
|---|---:|
| SFT относительно Foundation | {format_value(sft_difference)} п.п. |
| DPO относительно SFT | {format_value(dpo_difference)} п.п. |
| DPO относительно Foundation | {format_value(total_difference)} п.п. |
"""
else:
    comparison_text = "Недостаточно данных для сравнения двух групп."

table_text = summary.to_markdown(index=False)

bonus_md = f"""# Бонусное мини-исследование: влияние типа изображения на дообучение

## 1. Гипотеза

Эффект дообучения мультимодальной модели зависит от типа изображения.
Предполагается, что общий прирост точности после SFT и DPO относительно
Foundation будет выше на фотографиях реальных объектов, чем на рисунках
и схемах.

## 2. Протокол эксперимента

Для исследования использованы две группы изображений из `id_test`:

- Фотографии: {len(photo_ids)} заданных ID.
- Рисунки: {len(drawing_ids)} заданных ID.

Для каждого примера сравнивались три варианта модели:

- Foundation — базовая модель без адаптера.
- SFT (Reference) — модель после supervised fine-tuning.
- DPO (Aligned) — модель после preference alignment.

Для всех вариантов использовались одинаковые вопросы и изображения.
Качество оценивалось по точности выбора правильного ответа (Exact Match
по буквам A-E).

## 3. Наблюдения

Найдено примеров: {len(found_ids)} из {len(expected_ids)}.

{table_text}

### Разница между эффектами на двух группах

{comparison_text}

## 4. Вывод

{conclusion}

Результаты являются предварительными: выборки небольшие, поэтому
наблюдаемая разница не доказывает универсального преимущества одного
типа изображений.

## 5. Ограничения

- В исследовании использована небольшая выборка из 44 примеров.
- Результаты зависят от конкретных выбранных изображений и вопросов.
- Exact Match оценивает правильность ответа, но не объяснение модели.
- Отсутствие статистической проверки не позволяет делать вывод
  о статистической значимости различий.
- Для выполнения строгого протокола бонусного задания дополнительные
  входы должны быть отделены от отложенных тестовых выборок.

## 6. Файлы с результатами

- `bonus_photo_drawing_details.csv` — предсказания по каждому примеру.
- `bonus_photo_drawing_summary.csv` — сводные метрики по группам.
"""

bonus_path = ARTIFACTS / "bonus.md"
bonus_path.write_text(bonus_md, encoding="utf-8")

display(Markdown("### Предварительный вывод"))
display(Markdown(conclusion))

print("\nРезультаты сохранены:")
print(details_path)
print(summary_path)
print(bonus_path)

Найдено примеров: 44 из 44


## Бонусное исследование: фотографии и рисунки

**Гипотеза:** эффект дообучения SFT и последующего DPO на точность ответов зависит от типа изображения. Предполагается, что общий прирост точности на фотографиях будет выше, чем на рисунках.

### Сравнение точности

,Тип изображения,Примеров,"Foundation, %","SFT, %","DPO, %","Прирост SFT, п.п.","Прирост DPO, п.п.","Общий прирост, п.п."
0,Фотографии,22,81.82,86.36,81.82,4.55,-4.55,0.00
1,Рисунки,22,86.36,81.82,81.82,-4.55,0.00,-4.55


...

### Различия между группами

,Этап,"Разница прироста (фото − рисунки), п.п."
0,SFT относительно Foundation,9.10
1,DPO относительно SFT,-4.55
2,DPO относительно Foundation,4.55


### Предварительный вывод

Общий эффект SFT и DPO оказался выше на фотографиях. Результаты согласуются с гипотезой.


Результаты сохранены:
/content/artifacts/bonus_photo_drawing_details.csv
/content/artifacts/bonus_photo_drawing_summary.csv
/content/artifacts/bonus.md


In [25]:
from pathlib import Path
import shutil

# Папка с результатами эксперимента
ARTIFACTS = Path("artifacts").resolve()

assert ARTIFACTS.exists() and ARTIFACTS.is_dir(), \
    f"Папка artifacts не найдена: {ARTIFACTS}"

# Создаём архив рядом с папкой artifacts
archive_base = ARTIFACTS.parent / "artifacts"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=ARTIFACTS.parent,
    base_dir=ARTIFACTS.name
)

print("Архив успешно создан:", archive_path)
print(f"Размер архива: {Path(archive_path).stat().st_size / (1024 ** 2):.2f} MB")

# Скачивание в Google Colab
try:
    from google.colab import files
    files.download(archive_path)

except ImportError:
    # Для Jupyter Notebook отображаем ссылку на скачивание
    from IPython.display import display, FileLink
    display(FileLink(archive_path))

Архив успешно создан: /content/artifacts.zip
Размер архива: 635.44 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Анализ результатов

На общей выборке из 192 примеров Foundation показала точность 78,12%, Reference — 81,77%, Aligned — 81,25%. Таким образом, SFT повысило точность на 3,65 п.п., а **DPO сохранило большую часть прироста, но не дало дополнительного улучшения.**

По отдельным выборкам:

- ID Test: точность практически не изменилась: 87,50% у Foundation и Reference, 85,94% у Aligned.
- OOD Test: наблюдается наибольший прирост после SFT и DPO — с 57,81% до 68,75%.
- General Test: все три модели показали одинаковую точность — 89,06%.

Ручная оценка предпочтений. Среди 48 решающих сравнений Reference получила 35 побед (72,92%), Foundation — 13 (27,08%). В остальных 144 случаях была зафиксирована ничья. **Это указывает на преимущество Reference по ручной оценке.**

Бонусное исследование. SFT показало более положительный эффект на фотографиях (+4,55 п.п.), чем на рисунках (−4,55 п.п.). После DPO прирост относительно Foundation на фотографиях исчез, а на рисунках результат не изменился.

Итоговый вывод. Основной положительный эффект обеспечило SFT: повысилась точность, особенно на OOD Test, и улучшился формат ответов. **DPO не дало дополнительного прироста точности относительно Reference.** Результаты следует считать предварительными из-за небольшого размера выборок и отсутствия статистической проверки.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.